# FedVocabHAR WISDM Phase Zero-C: Conditional Calibration

Phase Zero-B established that the hybrid representation transfers to unseen activities (AUROC 0.8502), but one global threshold did not transfer across client pairs. This notebook keeps the Phase Zero-B representation fixed and tests the remaining hypothesis:

> A candidate-pair score should be judged relative to mismatched known-activity anchors from the **same two clients**, rather than against one global score distribution.

For every client pair, known anchors form a local null distribution. Raw compatibility is converted to a normalized margin and an empirical conformal percentile. Final novel labels are never used for calibration.

**Locked gate:** AUROC ≥ 0.85, TPR ≥ 0.70, FPR ≤ 0.05. The registry runs only if this gate passes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

@dataclass(frozen=True)
class Config:
    project: str='/content/drive/MyDrive/FedVocabHAR'
    seed: int=42
    known: tuple=('A','B','C','D','E','R')
    development_novel: tuple=('F','G','H','I','K')
    engineering_novel: tuple=('L','M','O','P','Q','S')
    development_subjects: tuple=tuple(range(1635,1642))
    engineering_subjects: tuple=tuple(range(1643,1651))
    sampling_rate: float=20.0
    embedding_dim: int=96
    target_fpr: float=.05
    registry_margin: float=.05
    registry_trials: int=100

CFG=Config(); PROJECT=Path(CFG.project); P0=PROJECT/'artifacts'/'phase0'; P0B=PROJECT/'artifacts'/'phase0b'; OUT=PROJECT/'artifacts'/'phase0c'
OUT.mkdir(parents=True,exist_ok=True); CACHE=P0/'wisdm_watch_accel_w128_s64.npz'; MODEL=P0B/'contrastive_encoder.pt'
assert CACHE.exists() and MODEL.exists(), 'Run Phase Zero-B first.'
random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',DEVICE)

device: cuda


In [3]:
d=np.load(CACHE); x=d['x'].astype(np.float32); subjects=d['subject']; activities=d['activity']
dev_mask=np.isin(subjects,CFG.development_subjects)

def signal_features(x,fs=20.0):
    mag=np.linalg.norm(x,axis=1,keepdims=True); z=np.concatenate([x,mag],axis=1)
    centered=z-z.mean(2,keepdims=True); std=centered.std(2)+1e-8; feats=[]
    funcs=[lambda a:a.mean(2),lambda a:a.std(2),lambda a:np.median(a,2),lambda a:np.quantile(a,.10,axis=2),
           lambda a:np.quantile(a,.25,axis=2),lambda a:np.quantile(a,.75,axis=2),lambda a:np.quantile(a,.90,axis=2),
           lambda a:np.sqrt((a*a).mean(2)),lambda a:np.abs(a).mean(2),lambda a:np.sqrt((np.diff(a,axis=2)**2).mean(2))]
    feats += [fn(z) for fn in funcs]
    feats.append((np.diff(np.signbit(centered),axis=2)!=0).mean(2))
    feats.append((centered[:,:,:-1]*centered[:,:,1:]).mean(2)/(std*std))
    fft=np.abs(np.fft.rfft(centered,axis=2))**2; freqs=np.fft.rfftfreq(z.shape[2],1/fs); fft[:,:,0]=0
    total=fft.sum(2)+1e-8; p=fft/total[:,:,None]
    feats += [(fft*freqs).sum(2)/total,freqs[np.argmax(fft,axis=2)],-(p*np.log(p+1e-12)).sum(2)]
    for lo,hi in [(0,.5),(.5,1.5),(1.5,3),(3,5),(5,10.1)]:
        band=(freqs>=lo)&(freqs<hi); feats.append(fft[:,:,band].sum(2)/total)
    feats.append(np.stack([(centered[:,0]*centered[:,1]).mean(1)/(std[:,0]*std[:,1]),
                           (centered[:,0]*centered[:,2]).mean(1)/(std[:,0]*std[:,2]),
                           (centered[:,1]*centered[:,2]).mean(1)/(std[:,1]*std[:,2])],1))
    return np.concatenate(feats,1).astype(np.float32)

sf=signal_features(x,CFG.sampling_rate); print('signal features:',sf.shape)

signal features: (19704, 83)


In [4]:
class Encoder(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.body=nn.Sequential(nn.Conv1d(3,64,7,padding=3),nn.BatchNorm1d(64),nn.ReLU(),
          nn.Conv1d(64,96,5,stride=2,padding=2),nn.BatchNorm1d(96),nn.ReLU(),
          nn.Conv1d(96,128,5,stride=2,padding=2),nn.BatchNorm1d(128),nn.ReLU(),
          nn.Conv1d(128,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.AdaptiveAvgPool1d(1))
        self.project=nn.Sequential(nn.Linear(128,128),nn.ReLU(),nn.Linear(128,dim))
    def forward(self,z): return F.normalize(self.project(self.body(z).squeeze(-1)),dim=1)

ckpt=torch.load(MODEL,map_location=DEVICE,weights_only=False); encoder=Encoder(CFG.embedding_dim).to(DEVICE)
encoder.load_state_dict(ckpt['state_dict']); encoder.eval(); xn=(x-ckpt['channel_mean'])/ckpt['channel_std']
emb=[]
with torch.no_grad():
    for start in range(0,len(xn),1024): emb.append(encoder(torch.from_numpy(xn[start:start+1024]).to(DEVICE)).cpu().numpy())
emb=np.concatenate(emb); print('embeddings:',emb.shape)

embeddings: (19704, 96)


## Build fixed candidates

The representation is identical to Phase Zero-B. We now include labeled known-activity anchors for both development and engineering client pairs, solely to estimate each pair's null compatibility distribution.

In [5]:
def aggregate(v,kind):
    if kind=='signal': return np.concatenate([np.median(v,0),v.mean(0),v.std(0)])
    mean=v.mean(0); mean/=np.linalg.norm(mean)+1e-12; return np.concatenate([mean,v.std(0)])

def candidates(subjs,acts):
    out={}
    for s in subjs:
        for a in acts:
            m=(subjects==s)&(activities==a)
            if m.sum()>=3: out[(int(s),str(a))]={'signal':aggregate(sf[m],'signal'),'ssl':aggregate(emb[m],'ssl'),'support':int(m.sum())}
    return out

dev=candidates(CFG.development_subjects,CFG.development_novel); dev_known=candidates(CFG.development_subjects,CFG.known)
eng=candidates(CFG.engineering_subjects,CFG.engineering_novel); eng_known=candidates(CFG.engineering_subjects,CFG.known)
for view in ('signal','ssl'):
    vals=np.stack([v[view] for v in dev.values()]); mu=vals.mean(0); sd=vals.std(0)+1e-8
    for pool in (dev,dev_known,eng,eng_known):
        for item in pool.values(): item[view]=(item[view]-mu)/sd

sw=len(next(iter(dev.values()))['signal'])//3; zw=len(next(iter(dev.values()))['ssl'])//2
def metrics(a,b,widths):
    out=[]; start=0
    for width in widths:
        u=a[start:start+width]; v=b[start:start+width]; d=u-v; start+=width
        out += [np.mean(np.abs(d)),np.sqrt(np.mean(d*d)),np.max(np.abs(d)),np.dot(u,v)/(np.linalg.norm(u)*np.linalg.norm(v)+1e-12),
                np.corrcoef(u,v)[0,1] if np.std(u)>0 and np.std(v)>0 else 0.]
    return np.nan_to_num(np.asarray(out,dtype=np.float32))
def pair_features(a,b):
    return np.r_[metrics(a['signal'],b['signal'],[sw]*3),metrics(a['ssl'],b['ssl'],[zw]*2),
                 abs(math.log1p(a['support'])-math.log1p(b['support'])),math.log1p(min(a['support'],b['support']))]
def make_pairs(pool,subjs,acts):
    rows=[]
    for i,s1 in enumerate(subjs):
        for s2 in subjs[i+1:]:
            for a in acts:
                for b in acts:
                    if (s1,a) in pool and (s2,b) in pool: rows.append((pool[(s1,a)],pool[(s2,b)],int(a==b),int(s1),int(s2),str(a),str(b)))
    return rows
dev_pairs=make_pairs(dev,list(CFG.development_subjects),list(CFG.development_novel))
eng_pairs=make_pairs(eng,list(CFG.engineering_subjects),list(CFG.engineering_novel))

## Client-pair conditional score

For clients *i* and *j*, the 30 mismatched known-anchor comparisons estimate the null distribution. The raw logit is centered at that pair's 95th null percentile and scaled by the gap to the median same-known-activity score. This creates a comparable margin across heterogeneous client pairs. We also retain an empirical null percentile for audit.

In [6]:
def make_model(): return Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(max_iter=4000,class_weight='balanced',C=.1))])
def logit(p): return np.log(np.clip(p,1e-6,1-1e-6)/np.clip(1-p,1e-6,1))

def anchor_scores(model,known_pool,s1,s2):
    pos=[]; neg=[]
    for a in CFG.known:
        for b in CFG.known:
            p=float(model.predict_proba(pair_features(known_pool[(s1,a)],known_pool[(s2,b)])[None])[0,1])
            (pos if a==b else neg).append(p)
    return np.asarray(pos),np.asarray(neg)

def conditional(raw,pos,neg):
    q=float(np.quantile(neg,.95,method='higher')); center=logit(q)
    upper=float(np.median(logit(pos))); spread=max(upper-center,float(np.subtract(*np.percentile(logit(neg),[75,25]))),.15)
    margin=float((logit(raw)-center)/spread)
    percentile=float((1+(neg<raw).sum())/(len(neg)+1))
    return margin,percentile,q

oof=[]
for held in CFG.development_novel:
    train=[r for r in dev_pairs if r[5]!=held and r[6]!=held]; valid=[r for r in dev_pairs if r[5]==held or r[6]==held]
    X=np.stack([pair_features(a,b) for a,b,*_ in train]); y=np.array([r[2] for r in train]); fold=make_model().fit(X,y)
    cache={}
    for a,b,label,s1,s2,la,lb in valid:
        if (s1,s2) not in cache: cache[(s1,s2)]=anchor_scores(fold,dev_known,s1,s2)
        raw=float(fold.predict_proba(pair_features(a,b)[None])[0,1]); margin,pct,q=conditional(raw,*cache[(s1,s2)])
        oof.append({'label':label,'margin':margin,'percentile':pct,'raw':raw,'held_activity':held,'s1':s1,'s2':s2})
oof=pd.DataFrame(oof)
margin_threshold=float(np.quantile(oof.loc[oof.label==0,'margin'],.95,method='higher'))
pct_threshold=float(np.quantile(oof.loc[oof.label==0,'percentile'],.95,method='higher'))
print('development thresholds:',margin_threshold,pct_threshold)

development thresholds: 0.183789673814229 0.967741935483871


In [7]:
X=np.stack([pair_features(a,b) for a,b,*_ in dev_pairs]); y=np.array([r[2] for r in dev_pairs]); model=make_model().fit(X,y)
cache={}; predictions=[]
for a,b,label,s1,s2,la,lb in eng_pairs:
    if (s1,s2) not in cache: cache[(s1,s2)]=anchor_scores(model,eng_known,s1,s2)
    raw=float(model.predict_proba(pair_features(a,b)[None])[0,1]); margin,pct,q=conditional(raw,*cache[(s1,s2)])
    predictions.append({'label':label,'s1':s1,'s2':s2,'left_activity':la,'right_activity':lb,'raw':raw,'margin':margin,'percentile':pct,'pair_null_q95':q})
pred=pd.DataFrame(predictions); y=pred.label.to_numpy()
rows=[]
for score,threshold in [('margin',margin_threshold),('percentile',pct_threshold)]:
    p=pred[score].to_numpy(); rows.append({'score':score,'auroc':roc_auc_score(y,p),'average_precision':average_precision_score(y,p),
      'threshold':threshold,'tpr':float((p[y==1]>=threshold).mean()),'fpr':float((p[y==0]>=threshold).mean())})
results=pd.DataFrame(rows); display(results)
pred.to_csv(OUT/'pair_predictions.csv',index=False); oof.to_csv(OUT/'development_oof.csv',index=False)

,score,auroc,average_precision,threshold,tpr,fpr
0,margin,0.838981,0.555902,0.183790,0.672619,0.155952
1,percentile,0.838152,0.436660,0.967742,0.726190,0.190476


## Conditional registry audit

The registry uses the conditional margin for each pair of contributing clients. Complete-link compatibility requires every cross-client comparison to clear the locked development threshold.

In [8]:
primary=results.query("score=='margin'").iloc[0]
pair_gate=bool(primary.auroc>=.85 and primary.tpr>=.70 and primary.fpr<=.05)

def compat(a,s1,b,s2):
    key=tuple(sorted((s1,s2)))
    if key not in cache: cache[key]=anchor_scores(model,eng_known,*key)
    raw=float(model.predict_proba(pair_features(a,b)[None])[0,1]); return conditional(raw,*cache[key])[0]

def mix(a,b): return {'signal':.5*a['signal']+.5*b['signal'],'ssl':.5*a['ssl']+.5*b['ssl'],'support':min(a['support'],b['support'])}
def trial(mode,seed,quorum=3):
    rng=np.random.default_rng(seed); reports=[]
    for act in CFG.engineering_novel:
        for s in CFG.engineering_subjects: reports.append({'client':s,'truth':act,'sig':eng[(s,act)]})
    for s in CFG.engineering_subjects:
        if mode=='independent':
            a=rng.choice(CFG.known); sig=eng_known[(s,a)]; truth=f'false-known-{a}'
        else:
            a,b=rng.choice(CFG.engineering_novel,2,replace=False); sig=mix(eng[(s,a)],eng[(s,b)]); truth=f'mixture-{a}-{b}'
        reports.append({'client':s,'truth':truth,'sig':sig})
    rng.shuffle(reports); entries=[]; ambiguous=0
    for report in reports:
        choices=[]
        for idx,e in enumerate(entries):
            if report['client'] in {r['client'] for r in e}: continue
            scores=[compat(report['sig'],report['client'],r['sig'],r['client']) for r in e]
            if min(scores)>=margin_threshold: choices.append((float(np.mean(scores)),idx))
        choices.sort(reverse=True)
        if choices and (len(choices)==1 or choices[0][0]-choices[1][0]>=CFG.registry_margin): entries[choices[0][1]].append(report)
        else:
            if len(choices)>1: ambiguous+=1
            entries.append([report])
    promoted=[e for e in entries if len(e)>=quorum]; correct=0; recovered=set(); false=0
    for e in promoted:
        truths={r['truth'] for r in e}; real={t for t in truths if len(t)==1}
        if len(real)==1 and len(truths)==1: correct+=1; recovered|=real
        else: false+=1
    return {'precision':correct/len(promoted) if promoted else 0.,'recall':len(recovered)/len(CFG.engineering_novel),
            'false_merge_rate':false/len(promoted) if promoted else 0.,'promoted':len(promoted),'ambiguous':ambiguous}

registry=pd.DataFrame()
if pair_gate:
    rr=[]
    for mode in ('independent','mixture'):
        ts=[trial(mode,CFG.seed+i) for i in range(CFG.registry_trials)]
        rr.append({'mode':mode,**{k:float(np.mean([t[k] for t in ts])) for k in ts[0]}})
    registry=pd.DataFrame(rr); display(registry)
else: print('Registry skipped: conditional pair gate did not pass.')

Registry skipped: conditional pair gate did not pass.


In [9]:
registry_gate=False
if not registry.empty:
    r=registry.query("mode=='independent'").iloc[0]; registry_gate=bool(r.precision>=.90 and r.recall>=.70)
decision={'pair_gate':pair_gate,'registry_gate':registry_gate,'proceed_to_full_implementation':bool(pair_gate and registry_gate)}
results.to_csv(OUT/'conditional_results.csv',index=False); registry.to_csv(OUT/'registry_results.csv',index=False)
manifest={'config':asdict(CFG),'decision':decision,'method':'client-pair conditional known-anchor calibration',
          'timestamp_utc':pd.Timestamp.utcnow().isoformat()}
(OUT/'phase0c_manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps(manifest,indent=2)); print('saved:',OUT)

{
  "config": {
    "project": "/content/drive/MyDrive/FedVocabHAR",
    "seed": 42,
    "known": [
      "A",
      "B",
      "C",
      "D",
      "E",
      "R"
    ],
    "development_novel": [
      "F",
      "G",
      "H",
      "I",
      "K"
    ],
    "engineering_novel": [
      "L",
      "M",
      "O",
      "P",
      "Q",
      "S"
    ],
    "development_subjects": [
      1635,
      1636,
      1637,
      1638,
      1639,
      1640,
      1641
    ],
    "engineering_subjects": [
      1643,
      1644,
      1645,
      1646,
      1647,
      1648,
      1649,
      1650
    ],
    "sampling_rate": 20.0,
    "embedding_dim": 96,
    "target_fpr": 0.05,
    "registry_margin": 0.05,
    "registry_trials": 100
  },
  "decision": {
    "pair_gate": false,
    "registry_gate": false,
    "proceed_to_full_implementation": false
  },
  "method": "client-pair conditional known-anchor calibration",
  "timestamp_utc": "2026-09-15T14:40:01.557037+00:00"
}
saved: /content

## Decision rule

- If the conditional pair gate and registry gate pass, proceed to the full task-free pipeline.
- If AUROC remains high but FPR remains above 0.05, the representation is viable but six known anchors are insufficient for a strict distribution-free threshold. The paper should use calibrated abstention with an explicit risk-coverage curve rather than claim fixed 5% FPR.
- If AUROC falls below 0.85, stop WISDM-only development and add a broader public development corpus before changing registry rules.